In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd
import requests


df_mapa_2024 = pd.read_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/final/dataset_mapa_2024.csv")
df_mapa_2024.head()


,provincia_id,provincia_nombre,departamento_id,departamento_nombre,anio,delitos_propiedad_hechos,tasa_delitos_propiedad_100k,tasa_delitos_propiedad_pct
0,2,ciudad autónoma de buenos aires,2001,comuna 1,2024,24297,10994.067900,10.994068
1,2,ciudad autónoma de buenos aires,2002,comuna 2,2024,8829,5497.201278,5.497201
2,2,ciudad autónoma de buenos aires,2003,comuna 3,2024,13277,6860.186941,6.860187
3,2,ciudad autónoma de buenos aires,2004,comuna 4,2024,15149,6672.862781,6.672863
4,2,ciudad autónoma de buenos aires,2005,comuna 5,2024,9219,4790.360043,4.790360


In [3]:
# Ver que provincias tiene el dataset
df_mapa_2024[["provincia_id", "provincia_nombre"]].drop_duplicates().sort_values("provincia_id")


,provincia_id,provincia_nombre
0,2,ciudad autónoma de buenos aires
15,6,buenos aires
150,10,catamarca
166,14,córdoba
192,18,corrientes
217,22,chaco
242,26,chubut
256,30,entre ríos
273,34,formosa
282,38,jujuy


In [4]:
# Descargar los centroides desde georef
centroides = []

provincias = (
    df_mapa_2024["provincia_id"]
    .drop_duplicates()
    .astype(int)
    .astype(str)
    .tolist()
)

for prov_id in provincias:
    url = "https://apis.datos.gob.ar/georef/api/departamentos"
    params = {
        "provincia": prov_id,
        "campos": "id,nombre,provincia,centroide",
        "max": 500
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    for d in data.get("departamentos", []):
        centroides.append({
            "departamento_id_geo": int(d["id"]),
            "departamento_geo": d["nombre"],
            "provincia_geo": d["provincia"]["nombre"],
            "lat": d["centroide"]["lat"],
            "lon": d["centroide"]["lon"]
        })

df_centroides = pd.DataFrame(centroides)
df_centroides.head()

,departamento_id_geo,departamento_geo,provincia_geo,lat,lon
0,2042,Comuna 6,Ciudad Autónoma de Buenos Aires,-34.616843,-58.443568
1,2007,Comuna 1,Ciudad Autónoma de Buenos Aires,-34.606444,-58.371512
2,2028,Comuna 4,Ciudad Autónoma de Buenos Aires,-34.642113,-58.387561
3,2035,Comuna 5,Ciudad Autónoma de Buenos Aires,-34.617370,-58.420572
4,2084,Comuna 12,Ciudad Autónoma de Buenos Aires,-34.566228,-58.490428


In [5]:
# Verifico que se descargaron bien los centroides
print(df_centroides.shape)
print(df_centroides.isna().sum())
df_centroides.sample(10, random_state=15)

(529, 5)
departamento_id_geo    0
departamento_geo       0
provincia_geo          0
lat                    0
lon                    0
dtype: int64


,departamento_id_geo,departamento_geo,provincia_geo,lat,lon
44,6196,Coronel Pringles,Buenos Aires,-38.147917,-61.264417
216,18105,Mercedes,Corrientes,-29.064507,-57.818386
73,6147,Carlos Casares,Buenos Aires,-35.749922,-61.374378
124,6105,Bolívar,Buenos Aires,-36.298948,-61.149860
436,70070,Pocito,San Juan,-31.747254,-68.584680
503,86189,Silípica,Santiago del Estero,-28.188849,-64.273268
367,54119,25 de Mayo,Misiones,-27.378614,-54.634090
189,14105,Río Primero,Córdoba,-31.031524,-63.436115
125,6063,Balcarce,Buenos Aires,-37.714620,-58.271748
495,86161,Robles,Santiago del Estero,-27.853944,-63.907509


In [6]:
# Guardar la biblioteca de centroides
df_centroides.to_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/processed/georef/centroides.csv", index=False)

In [7]:
# Unir los centroides con el dataset del mapa
df_mapa_2024 = df_mapa_2024.merge(df_centroides, left_on=["departamento_id"],
right_on=["departamento_id_geo"], how="left")

In [8]:
# ELiminar columna duplicada
df_mapa_2024 = df_mapa_2024.drop(columns=["departamento_id_geo"], errors="ignore")

In [9]:
# verifico nulos
df_mapa_2024.isna().sum()

,0
provincia_id,0
provincia_nombre,0
departamento_id,0
departamento_nombre,0
anio,0
delitos_propiedad_hechos,0
tasa_delitos_propiedad_100k,0
tasa_delitos_propiedad_pct,0
departamento_geo,17
provincia_geo,17


In [10]:
faltantes_geo = df_mapa_2024[df_mapa_2024["lat"].isna()][
    ["provincia_nombre", "departamento_nombre", "departamento_id"]
].drop_duplicates()

faltantes_geo

,provincia_nombre,departamento_nombre,departamento_id
0,ciudad autónoma de buenos aires,comuna 1,2001
1,ciudad autónoma de buenos aires,comuna 2,2002
2,ciudad autónoma de buenos aires,comuna 3,2003
3,ciudad autónoma de buenos aires,comuna 4,2004
4,ciudad autónoma de buenos aires,comuna 5,2005
5,ciudad autónoma de buenos aires,comuna 6,2006
7,ciudad autónoma de buenos aires,comuna 8,2008
8,ciudad autónoma de buenos aires,comuna 9,2009
9,ciudad autónoma de buenos aires,comuna 10,2010
10,ciudad autónoma de buenos aires,comuna 11,2011


In [11]:
df_mapa_2024[df_mapa_2024["lat"].isna()]

,provincia_id,provincia_nombre,departamento_id,departamento_nombre,anio,delitos_propiedad_hechos,tasa_delitos_propiedad_100k,tasa_delitos_propiedad_pct,departamento_geo,provincia_geo,lat,lon
0,2,ciudad autónoma de buenos aires,2001,comuna 1,2024,24297,10994.067900,10.994068,NaN,NaN,NaN,NaN
1,2,ciudad autónoma de buenos aires,2002,comuna 2,2024,8829,5497.201278,5.497201,NaN,NaN,NaN,NaN
2,2,ciudad autónoma de buenos aires,2003,comuna 3,2024,13277,6860.186941,6.860187,NaN,NaN,NaN,NaN
3,2,ciudad autónoma de buenos aires,2004,comuna 4,2024,15149,6672.862781,6.672863,NaN,NaN,NaN,NaN
4,2,ciudad autónoma de buenos aires,2005,comuna 5,2024,9219,4790.360043,4.790360,NaN,NaN,NaN,NaN
5,2,ciudad autónoma de buenos aires,2006,comuna 6,2024,8297,4112.230130,4.112230,NaN,NaN,NaN,NaN
7,2,ciudad autónoma de buenos aires,2008,comuna 8,2024,9835,4823.726752,4.823727,NaN,NaN,NaN,NaN
8,2,ciudad autónoma de buenos aires,2009,comuna 9,2024,8878,5287.419301,5.287419,NaN,NaN,NaN,NaN
9,2,ciudad autónoma de buenos aires,2010,comuna 10,2024,8161,4747.638107,4.747638,NaN,NaN,NaN,NaN
10,2,ciudad autónoma de buenos aires,2011,comuna 11,2024,8889,4402.565563,4.402566,NaN,NaN,NaN,NaN


In [12]:
# ver que Ids no matchearon con georef
faltantes_ids = df_mapa_2024[df_mapa_2024["lat"].isna()]["departamento_id"].unique()
faltantes_ids

array([ 2001,  2002,  2003,  2004,  2005,  2006,  2008,  2009,  2010,
        2011,  2012,  2013,  2015,  6058,  6217, 94007, 94014])

In [13]:
# Comparo con los departamentos que tengo en el dataset de georef
ids_geo = df_centroides["departamento_id_geo"].unique()

set(faltantes_ids) - set(ids_geo)

{np.int64(2001),
 np.int64(2002),
 np.int64(2003),
 np.int64(2004),
 np.int64(2005),
 np.int64(2006),
 np.int64(2008),
 np.int64(2009),
 np.int64(2010),
 np.int64(2011),
 np.int64(2012),
 np.int64(2013),
 np.int64(2015),
 np.int64(6058),
 np.int64(6217),
 np.int64(94007),
 np.int64(94014)}

In [14]:
import requests

# Ejemplo de consulta a la API de georef para obtener información de un departamento específico
url = "https://apis.datos.gob.ar/georef/api/departamentos"
params = {
    "nombre": "ushuaia",
    "max": 10
}

r = requests.get(url, params=params)
r.json()

{'cantidad': 1,
 'departamentos': [{'centroide': {'lat': -54.7704676359158,
    'lon': -66.6836200601613},
   'id': '94015',
   'nombre': 'Ushuaia',
   'provincia': {'id': '94',
    'nombre': 'Tierra del Fuego, Antártida e Islas del Atlántico Sur'}}],
 'inicio': 0,
 'parametros': {'max': 10, 'nombre': 'ushuaia'},
 'total': 1}

In [15]:
# Para estos casos, no uso id. Uso fallback de nombre de departamento y provincia para obtener el centroide.
# Esto es porque algunos departamentos no tienen id en georef, pero si nombre.

# Separo los casos faltantes
df_faltantes = df_mapa_2024[df_mapa_2024["lat"].isna()].copy()



In [16]:
# Normalizo columnas antes de merge por nombre
for col in ["departamento_nombre", "provincia_nombre"]:
    df_mapa_2024[col] = df_mapa_2024[col].str.lower()

for col in ["departamento_geo", "provincia_geo"]:
    df_centroides[col] = df_centroides[col].str.lower()

In [17]:
# Hago merge por nombre de departamento y provincia
df_fallback = df_faltantes.merge(df_centroides, left_on=["provincia_nombre", "departamento_nombre"],
right_on=["provincia_geo", "departamento_geo"], how="left")

In [18]:
print(df_mapa_2024.columns.tolist())
print(df_fallback.columns.tolist())

['provincia_id', 'provincia_nombre', 'departamento_id', 'departamento_nombre', 'anio', 'delitos_propiedad_hechos', 'tasa_delitos_propiedad_100k', 'tasa_delitos_propiedad_pct', 'departamento_geo', 'provincia_geo', 'lat', 'lon']
['provincia_id', 'provincia_nombre', 'departamento_id', 'departamento_nombre', 'anio', 'delitos_propiedad_hechos', 'tasa_delitos_propiedad_100k', 'tasa_delitos_propiedad_pct', 'departamento_geo_x', 'provincia_geo_x', 'lat_x', 'lon_x', 'departamento_id_geo', 'departamento_geo_y', 'provincia_geo_y', 'lat_y', 'lon_y']


In [19]:
# quedarme con las columnas de latitud y longitud necesarias (que vienen del dataset de centroides) para actualizar el dataset original
df_fallback_limpio = df_fallback[[
    "provincia_nombre",
    "departamento_nombre",
    "lat_y",
    "lon_y"
]].copy()

df_fallback_limpio = df_fallback_limpio.rename(columns={
    "lat_y": "lat_fallback",
    "lon_y": "lon_fallback"
})

In [20]:
# Completar los nulos con los valores obtenidos por fallback
df_mapa_2024["lat"] = df_mapa_2024["lat"].fillna(df_fallback_limpio["lat_fallback"])
df_mapa_2024["lon"] = df_mapa_2024["lon"].fillna(df_fallback_limpio["lon_fallback"])

In [21]:
# Elimino las columnas auxiliares del merge
df_mapa_2024 = df_mapa_2024.drop(
    columns=["lat_fallback", "lon_fallback"],
    errors="ignore"
)

In [22]:
# Verifico nulos
df_mapa_2024 [df_mapa_2024["lat"].isna()][["provincia_nombre", "departamento_nombre", "departamento_id"]]

,provincia_nombre,departamento_nombre,departamento_id
23,buenos aires,quilmes,6058
46,buenos aires,chascomús,6217
520,"tierra del fuego, antártida e islas del atlánt...",río grande,94007
521,"tierra del fuego, antártida e islas del atlánt...",ushuaia,94014


In [23]:
# Reviso como se nombran los departamentos en georef para hacer consultas puntuales

for nombre in ["quilmes", "chascomus", "rio grande", "ushuaia"]:
    r = requests.get(
        "https://apis.datos.gob.ar/georef/api/departamentos",
        params={"nombre": nombre, "max": 5},
        timeout=30
    )
    print("\n", nombre.upper())
    print(r.json())


 QUILMES
{'cantidad': 1, 'departamentos': [{'centroide': {'lat': -34.7349707991152, 'lon': -58.2768580209942}, 'id': '06658', 'nombre': 'Quilmes', 'provincia': {'id': '06', 'nombre': 'Buenos Aires'}}], 'inicio': 0, 'parametros': {'max': 5, 'nombre': 'quilmes'}, 'total': 1}

 CHASCOMUS
{'cantidad': 1, 'departamentos': [{'centroide': {'lat': -35.6186883086057, 'lon': -57.9039810685242}, 'id': '06218', 'nombre': 'Chascomús', 'provincia': {'id': '06', 'nombre': 'Buenos Aires'}}], 'inicio': 0, 'parametros': {'max': 5, 'nombre': 'chascomus'}, 'total': 1}

 RIO GRANDE
{'cantidad': 1, 'departamentos': [{'centroide': {'lat': -53.7462919871752, 'lon': -68.1361046544505}, 'id': '94008', 'nombre': 'Río Grande', 'provincia': {'id': '94', 'nombre': 'Tierra del Fuego, Antártida e Islas del Atlántico Sur'}}], 'inicio': 0, 'parametros': {'max': 5, 'nombre': 'rio grande'}, 'total': 1}

 USHUAIA
{'cantidad': 1, 'departamentos': [{'centroide': {'lat': -54.7704676359158, 'lon': -66.6836200601613}, 'id': '

In [24]:
# Sabiendo los datos exactos, construyo una tabla manual para mergear con el dataset original y completar los nulos restantes

df_manual = pd.DataFrame([
    {
        "provincia_nombre": "buenos aires",
        "departamento_nombre": "quilmes",
        "lat_manual": -34.7349707991152,
        "lon_manual": -58.2768580209942
    },
    {
        "provincia_nombre": "buenos aires",
        "departamento_nombre": "chascomús",
        "lat_manual": -35.6186883086057,
        "lon_manual": -57.9039810685242
    },
    {
        "provincia_nombre": "tierra del fuego, antártida e islas del atlántico sur",
        "departamento_nombre": "río grande",
        "lat_manual": -53.7462919871752,
        "lon_manual": -68.1361046544505
    },
    {
        "provincia_nombre": "tierra del fuego, antártida e islas del atlántico sur",
        "departamento_nombre": "ushuaia",
        "lat_manual": -54.7704676359158,
        "lon_manual": -66.6836200601613
    }
])


In [25]:
# Merge con dataset original para completar los nulos restantes
df_mapa_2024 = df_mapa_2024.merge(df_manual, on=["provincia_nombre", "departamento_nombre"], how="left")

In [26]:
# Completo los datos nulos con los valores manuales
df_mapa_2024["lat"] = df_mapa_2024["lat"].fillna(df_mapa_2024["lat_manual"])
df_mapa_2024["lon"] = df_mapa_2024["lon"].fillna(df_mapa_2024["lon_manual"])

In [27]:
# Limpio las columnas auxiliares del merge manual
df_mapa_2024 = df_mapa_2024.drop(columns=["lat_manual", "lon_manual"], errors= "coerce")

In [28]:
# Verifico nulos
df_mapa_2024[["lat", "lon"]].isna().sum()

,0
lat,0
lon,0


In [29]:
# Crear dataframe con todos los años
# Verifico columnas en df_modelo
import pandas as pd
df_modelo = pd.read_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/final/df_modelo.csv")
df_modelo.columns.tolist()

['provincia_id',
 'provincia_nombre',
 'departamento_id',
 'departamento_nombre',
 'anio',
 'delitos_propiedad_hechos',
 'poblacion_2022',
 'tasa_delitos_propiedad_100k',
 'tasa_delitos_propiedad_pct',
 'region',
 'variacion_anual_salarios_pct',
 'variacion_anual_cbt',
 'variacion_empleo_const_pct',
 'mes',
 'variacion_anual_ipim',
 'jurisdiccion',
 'variacion_acceso_internet_pct',
 'Indice_IPC']

In [30]:
df_mapa = df_modelo.copy()
# primer merge (por id)
df_mapa = df_mapa.merge(df_centroides,
left_on="departamento_id", right_on="departamento_id_geo", how="left")

df_mapa[["lat", "lon"]].isna().sum()

,0
lat,130
lon,130


In [31]:
# Normalizar columnas de nombre de provincia y departamento para merge por nombre.
import unicodedata

def normalizar(s):
    if pd.isna(s):
        return s
    s = str(s).lower().strip()
    s = ''.join(
        c for c in unicodedata.normalize('NFKD', s)
        if not unicodedata.combining(c)
    )
    s = ' '.join(s.split())
    return s

df_mapa["provincia_key"] = df_mapa["provincia_nombre"].apply(normalizar)
df_mapa["departamento_key"] = df_mapa["departamento_nombre"].apply(normalizar)

df_centroides["provincia_key"] = df_centroides["provincia_geo"].apply(normalizar)
df_centroides["departamento_key"] = df_centroides["departamento_geo"].apply(normalizar)

In [32]:
# Ahora hago el fallback merge por nombre normalizado solo a los valores nulos de latitud y longitud
df_fallback = df_mapa[df_mapa["lat"].isna()].merge(
    df_centroides[["provincia_key", "departamento_key", "lat", "lon"]],
    on=["provincia_key", "departamento_key"],
    how="left",
    suffixes=("", "_geo")
)


In [33]:
df_fallback.columns.tolist()

['provincia_id',
 'provincia_nombre',
 'departamento_id',
 'departamento_nombre',
 'anio',
 'delitos_propiedad_hechos',
 'poblacion_2022',
 'tasa_delitos_propiedad_100k',
 'tasa_delitos_propiedad_pct',
 'region',
 'variacion_anual_salarios_pct',
 'variacion_anual_cbt',
 'variacion_empleo_const_pct',
 'mes',
 'variacion_anual_ipim',
 'jurisdiccion',
 'variacion_acceso_internet_pct',
 'Indice_IPC',
 'departamento_id_geo',
 'departamento_geo',
 'provincia_geo',
 'lat',
 'lon',
 'provincia_key',
 'departamento_key',
 'lat_geo',
 'lon_geo']

In [34]:

df_mapa["lat"] = df_mapa["lat"].fillna(df_fallback["lat_geo"])
df_mapa["lon"] = df_mapa["lon"].fillna(df_fallback["lon_geo"])

In [35]:
df_mapa[["lat", "lon"]].isna().sum()

,0
lat,26
lon,26


In [36]:
# Completo los nulos restantes con el merge manual por nombre de provincia y departamento
#df_manual = pd.DataFrame([
#    {"provincia_nombre": "buenos aires", "departamento_nombre": "quilmes", "lat_manual": -34.73497, "lon_manual": -58.27685},
#    {"provincia_nombre": "buenos aires", "departamento_nombre": "chascomús", "lat_manual": -35.61868, "lon_manual": -57.90398},
#    {"provincia_nombre": "tierra del fuego, antártida e islas del atlántico sur", "departamento_nombre": "río grande", "lat_manual": -53.74629, "lon_manual": -68.13610},
#    {"provincia_nombre": "tierra del fuego, antártida e islas del atlántico sur", "departamento_nombre": "ushuaia", "lat_manual": -54.77046, "lon_manual": -66.68362}
#])

df_mapa = df_mapa.merge(
    df_manual,
    on=["provincia_nombre", "departamento_nombre"],
    how="left"
)

df_mapa["lat"] = df_mapa["lat"].fillna(df_mapa["lat_manual"])
df_mapa["lon"] = df_mapa["lon"].fillna(df_mapa["lon_manual"])

In [37]:

print(df_mapa[["lat", "lon"]].isna().sum())
print(df_mapa.columns.tolist())
print(df_mapa.shape)

lat    0
lon    0
dtype: int64
['provincia_id', 'provincia_nombre', 'departamento_id', 'departamento_nombre', 'anio', 'delitos_propiedad_hechos', 'poblacion_2022', 'tasa_delitos_propiedad_100k', 'tasa_delitos_propiedad_pct', 'region', 'variacion_anual_salarios_pct', 'variacion_anual_cbt', 'variacion_empleo_const_pct', 'mes', 'variacion_anual_ipim', 'jurisdiccion', 'variacion_acceso_internet_pct', 'Indice_IPC', 'departamento_id_geo', 'departamento_geo', 'provincia_geo', 'lat', 'lon', 'provincia_key', 'departamento_key', 'lat_manual', 'lon_manual']
(4072, 27)


In [38]:
# Borrar las columnas auxiliares de los merges
df_mapa = df_mapa.drop(columns=[
    "departamento_id_geo", "provincia_key", "departamento_key", "lat_geo", "lon_geo", "lat_manual", "lon_manual","departamento_geo", "provincia_geo"
], errors="ignore")

In [39]:
# Filtro los años con datos completos para el análisis de mapas
df_mapa_final = df_mapa[df_mapa["anio"]>=2017].copy()

In [40]:
# Renombrar departamentos "capital" en Mendoza y Córdoba para diferenciarlos porque figuran ambos como "capital" y eso genera confusión al hacer el merge con georef. Esto lo hago al final para no tener problemas de merge durante el proceso de georreferenciación.
df_mapa_final.loc[
    (df_mapa_final["provincia_nombre"] == "mendoza") & (df_mapa_final["departamento_nombre"] == "capital"),
    "departamento_nombre"
] = "mendoza capital"

df_mapa_final.loc[
    (df_mapa_final["provincia_nombre"] == "córdoba") & (df_mapa_final["departamento_nombre"] == "capital"),
    "departamento_nombre"
] = "cordoba capital"

In [41]:
df_mapa_final.isna().sum().sort_values(ascending=False)

,0
provincia_id,0
provincia_nombre,0
departamento_id,0
departamento_nombre,0
anio,0
delitos_propiedad_hechos,0
poblacion_2022,0
tasa_delitos_propiedad_100k,0
tasa_delitos_propiedad_pct,0
region,0


In [42]:
df_mapa_final.shape

(4072, 20)

In [43]:
# Guardar el dataset final con coordenadas para análisis de mapas
df_mapa_final.to_csv(r"/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/final/dataset_mapa_final.csv", index=False)

## Creación del Mapa de calor

In [44]:
path1 = r'/content/drive/MyDrive/Practica_Profesionalizante_3/Copia_pre_proyecto/data/final/dataset_mapa_sup_final.csv' # ACTUALIZA ESTA RUTA a la ubicación de tu archivodf = pd.read_csv(path)
df1= pd.read_csv(path1)

In [45]:
df1.columns.tolist()


['provincia_id',
 'provincia_nombre',
 'departamento_id',
 'departamento_nombre',
 'anio',
 'delitos_propiedad_hechos',
 'poblacion_2022',
 'tasa_delitos_propiedad_100k',
 'tasa_delitos_propiedad_pct',
 'region',
 'variacion_anual_salarios_pct',
 'variacion_anual_cbt',
 'variacion_empleo_const_pct',
 'mes',
 'variacion_anual_ipim',
 'jurisdiccion',
 'variacion_acceso_internet_pct',
 'Indice_IPC',
 'lat',
 'lon',
 'provincia_key',
 'departamento_key',
 'poblacion',
 'Superficie en km2',
 '_merge',
 'flag_poblacion_faltante',
 'tasa_delitos_propiedad_100k_v2',
 'densidad_poblacion',
 'tasa_lag1',
 'Indice_IPC_lag1',
 'variacion_anual_cbt_lag1',
 'variacion_anual_salarios_lag1',
 'tasa_yoy',
 'log_densidad']

In [46]:
# Filtro columnas para el gráfico
df_plot = df1[[
    "provincia_nombre",
    "departamento_nombre",
    "anio",
    "lat",
    "lon",
    "tasa_delitos_propiedad_100k_v2",
    "delitos_propiedad_hechos",
    "tasa_delitos_propiedad_pct"
]].copy()

df_plot["tasa_delitos_propiedad_100k_v2"] = df_plot["tasa_delitos_propiedad_100k_v2"].round(2)
df_plot["tasa_delitos_propiedad_pct"] = df_plot["tasa_delitos_propiedad_pct"].round(3)

In [47]:
df1 = df1.dropna(subset=["lat", "lon", "tasa_delitos_propiedad_100k_v2"])

In [48]:
# ============================================================
# MAPA INTERACTIVO EN FOLIUM
# ------------------------------------------------------------
# Esta versión incluye:
# - filtros interactivos por año y provincia
# - tamaño de círculos normalizado según la tasa
# - color de círculos según intensidad de la tasa
# - tooltip + popup detallado
# - título incrustado sobre el mapa
# - control de capas
# - cluster de marcadores para zonas densas
# - leyenda de color
# - leyenda visual de tamaño


import folium
import branca.colormap as cm
import ipywidgets as widgets

from folium.plugins import MarkerCluster
from IPython.display import display


# ============================================================
# 1. WIDGETS DE CONTROL
# ------------------------------------------------------------
# Se crean dos controles:
# - un slider para elegir el año
# - un dropdown para elegir una provincia o ver todo el país
# ============================================================

anio_widget = widgets.SelectionSlider(
    options=sorted(df1["anio"].dropna().unique()),
    value=2024,
    description="Año",
    continuous_update=False,
    layout=widgets.Layout(width="320px")
)

provincia_widget = widgets.Dropdown(
    options=["Todas"] + sorted(df1["provincia_nombre"].dropna().unique()),
    value="Todas",
    description="Provincia",
    layout=widgets.Layout(width="320px")
)

# Output donde se dibujará el mapa cada vez que cambie un filtro
output = widgets.Output()


# ============================================================
# 2. FUNCIONES AUXILIARES
# ============================================================

def calcular_radio(valor, vmin, vmax, rmin=4, rmax=16):
    """
    Escala el tamaño del círculo entre rmin y rmax
    usando normalización lineal sobre la variable elegida.
    """
    if vmax == vmin:
        return (rmin + rmax) / 2
    return rmin + ((valor - vmin) / (vmax - vmin)) * (rmax - rmin)


def formatear_numero(valor, decimales=2):
    """
    Formatea números para el popup de forma prolija.
    """
    return f"{valor:,.{decimales}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def agregar_titulo(mapa, anio, provincia):
    """
    Agrega un título HTML incrustado sobre el mapa.
    """
    subtitulo = "Total país" if provincia == "Todas" else provincia

    titulo_html = f"""
    <div style="
        position: fixed;
        top: 10px;
        left: 50px;
        z-index: 9999;
        background-color: rgba(17,17,17,0.88);
        color: white;
        padding: 10px 14px;
        border-radius: 8px;
        font-size: 16px;
        font-weight: bold;
        box-shadow: 0 0 6px rgba(0,0,0,0.45);
        border: 1px solid rgba(255,255,255,0.08);
    ">
        Tasa de delitos contra la propiedad por departamento - Año {anio}<br>
        <span style="font-size: 13px; font-weight: normal; color: #d0d0d0;">
            Cobertura: {subtitulo}
        </span>
    </div>
    """
    mapa.get_root().html.add_child(folium.Element(titulo_html))


def agregar_leyenda_tamanio(mapa):
    """
    Agrega una leyenda visual para interpretar el tamaño de los círculos.
    Es una referencia conceptual de 'menor' a 'mayor' tasa.
    """
    leyenda_html = """
    <div style="
        position: fixed;
        bottom: 30px;
        left: 30px;
        z-index: 9999;
        background-color: rgba(17,17,17,0.92);
        color: white;
        padding: 12px 14px;
        border-radius: 8px;
        font-size: 12px;
        box-shadow: 0 0 6px rgba(0,0,0,0.45);
        border: 1px solid rgba(255,255,255,0.08);
        width: 185px;
    ">
        <div style="font-weight: bold; margin-bottom: 8px;">
            Tamaño del marcador
        </div>

        <div style="display: flex; align-items: center; margin-bottom: 6px;">
            <div style="
                width: 8px; height: 8px; border-radius: 50%;
                background: #ffb74d; margin-right: 10px;
                border: 1px solid #111;
            "></div>
            <span>Menor tasa relativa</span>
        </div>

        <div style="display: flex; align-items: center; margin-bottom: 6px;">
            <div style="
                width: 14px; height: 14px; border-radius: 50%;
                background: #f4511e; margin-right: 10px;
                border: 1px solid #111;
            "></div>
            <span>Tasa intermedia</span>
        </div>

        <div style="display: flex; align-items: center;">
            <div style="
                width: 22px; height: 22px; border-radius: 50%;
                background: #8e0038; margin-right: 10px;
                border: 1px solid #111;
            "></div>
            <span>Mayor tasa relativa</span>
        </div>
    </div>
    """
    mapa.get_root().html.add_child(folium.Element(leyenda_html))


# ============================================================
# 3. FUNCIÓN PRINCIPAL DE CREACIÓN DEL MAPA
# ============================================================

def crear_mapa_folium_completo(df_base, anio, provincia="Todas", usar_cluster=True):

    # --------------------------------------------------------
    # A. FILTRADO DEL DATASET
    # --------------------------------------------------------
    df_filtered = df_base[df_base["anio"] == anio].copy()

    if provincia != "Todas":
        df_filtered = df_filtered[df_filtered["provincia_nombre"] == provincia].copy()

    # Filtrado defensivo para asegurar que las coordenadas y
    # la variable visual principal estén disponibles
    df_filtered = df_filtered.dropna(
        subset=["lat", "lon", "tasa_delitos_propiedad_100k_v2"]
    )

    if df_filtered.empty:
        return None

    # --------------------------------------------------------
    # B. CENTRO Y ZOOM DEL MAPA
    # --------------------------------------------------------
    # El centro se calcula dinámicamente sobre el subconjunto filtrado
    center_lat = df_filtered["lat"].mean()
    center_lon = df_filtered["lon"].mean()

    # Ajuste de zoom según el nivel geográfico observado
    zoom_base = 4 if provincia == "Todas" else 6

    # --------------------------------------------------------
    # C. CREACIÓN DEL MAPA BASE
    # --------------------------------------------------------
    mapa = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=zoom_base,
        tiles="CartoDB dark_matter",
        control_scale=True
    )

    # --------------------------------------------------------
    # D. ESCALA DE COLOR
    # --------------------------------------------------------
    tasa_min = df_filtered["tasa_delitos_propiedad_100k_v2"].min()
    tasa_max = df_filtered["tasa_delitos_propiedad_100k_v2"].max()

    colormap = cm.LinearColormap(
        colors=["#ffe082", "#ffb74d", "#ff7043", "#f4511e", "#d32f2f", "#8e0038"],
        vmin=tasa_min,
        vmax=tasa_max
    )
    colormap.caption = "Tasa de delitos contra la propiedad por 100.000 hab."
    colormap.add_to(mapa)

    # --------------------------------------------------------
    # E. CAPAS DEL MAPA
    # --------------------------------------------------------
    # Capa con los puntos individuales
    capa_puntos = folium.FeatureGroup(name="Departamentos", show=True)

    # Capa cluster opcional para zonas con mucha densidad
    if usar_cluster:
        cluster = MarkerCluster(name="Agrupación de marcadores").add_to(capa_puntos)

    # --------------------------------------------------------
    # F. RECORRIDO DE FILAS Y CREACIÓN DE MARCADORES
    # --------------------------------------------------------
    for _, row in df_filtered.iterrows():
        tasa = row["tasa_delitos_propiedad_100k_v2"]
        radio = calcular_radio(tasa, tasa_min, tasa_max, rmin=4, rmax=16)

        # Popup enriquecido
        popup_html = f"""
        <div style="font-family: Arial; font-size: 13px; width: 270px;">
            <h4 style="margin-bottom: 8px; color: #111;">
                {row['departamento_nombre']}
            </h4>

            <b>Provincia:</b> {row['provincia_nombre']}<br>
            <b>Año:</b> {row['anio']}<br>
            <b>Hechos registrados:</b> {int(row['delitos_propiedad_hechos']) if pd.notna(row['delitos_propiedad_hechos']) else 'N/D'}<br>
            <b>Tasa por 100.000 hab.:</b> {formatear_numero(row['tasa_delitos_propiedad_100k_v2'], 2)}<br>
            <b>Tasa porcentual:</b> {formatear_numero(row['tasa_delitos_propiedad_pct'], 3)}
        </div>
        """

        tooltip_text = (
            f"{row['departamento_nombre']} | "
            f"{row['provincia_nombre']} | "
            f"Tasa: {formatear_numero(row['tasa_delitos_propiedad_100k_v2'], 2)}"
        )

        marcador = folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=radio,
            popup=folium.Popup(popup_html, max_width=320),
            tooltip=tooltip_text,
            color="#111111",
            weight=0.7,
            fill=True,
            fill_color=colormap(tasa),
            fill_opacity=0.82
        )

        if usar_cluster:
            marcador.add_to(cluster)
        else:
            marcador.add_to(capa_puntos)

    # La capa final se agrega al mapa
    capa_puntos.add_to(mapa)

    # --------------------------------------------------------
    # G. ELEMENTOS DE INTERFAZ
    # --------------------------------------------------------
    folium.LayerControl(collapsed=False).add_to(mapa)
    agregar_titulo(mapa, anio, provincia)
    agregar_leyenda_tamanio(mapa)

    return mapa


# ============================================================
# 4. FUNCIÓN DE ACTUALIZACIÓN DEL OUTPUT
# ------------------------------------------------------------
# Cada vez que cambia el año o la provincia, se recrea el mapa
# y se vuelve a mostrar dentro del widget de salida.
# ============================================================

def actualizar_mapa(change=None):
    with output:
        output.clear_output(wait=True)

        mapa = crear_mapa_folium_completo(
            df_base=df1,
            anio=anio_widget.value,
            provincia=provincia_widget.value,
            usar_cluster=False                # Colocar "True" para activar la agrupación por cluster
        )

        if mapa is None:
            print("No hay datos para los filtros seleccionados.")
            return

        display(mapa)


# ============================================================
# 5. VINCULACIÓN DE EVENTOS
# ============================================================

anio_widget.observe(actualizar_mapa, names="value")
provincia_widget.observe(actualizar_mapa, names="value")


# ============================================================
# 6. DISPLAY FINAL EN EL NOTEBOOK
# ============================================================

display(widgets.HBox([anio_widget, provincia_widget]))
display(output)

# Primera renderización
actualizar_mapa()

Output()